In [1]:
"""
Step 4: Data Preparation for Modeling (UPDATED FOR AUTOREGRESSIVE FORECASTING)
======================================
Purpose: Create chronological time-series sequences for training step-by-step growth prediction models
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

In [2]:
# 1. LOAD ENGINEERED FEATURES

print("\n1. LOADING ENGINEERED FEATURES...")
print("-" * 80)

df = pd.read_csv('/Users/dilumsamarathunga/Projects/MachineLearning/child_growth_prediction/data/processed/features_engineered.csv')
df['date'] = pd.to_datetime(df['date'])
df['dob'] = pd.to_datetime(df['dob'])

print(f"✓ Loaded {len(df):,} records for {df['child_id'].nunique():,} children")



1. LOADING ENGINEERED FEATURES...
--------------------------------------------------------------------------------
✓ Loaded 310,850 records for 57,684 children


In [3]:
df = df.replace([np.inf, -np.inf], np.nan)

print("NaN counts before cleaning:")
nan_counts = df.isna().sum()
nan_counts = nan_counts[nan_counts > 0]
if len(nan_counts) > 0:
    print(nan_counts)
else:
    print("  No NaN values found")

critical_cols = ['height', 'weight', 'age_months', 'cbmi', 'gender_numeric']
before_drop = len(df)
df = df.dropna(subset=critical_cols)
after_drop = len(df)
print(f"\n✓ Dropped {before_drop - after_drop:,} rows with NaN in critical columns")

derived_cols = ['height_velocity_monthly', 'weight_velocity_monthly', 
                'zlen', 'zwei', 'zwfl', 'zbmi',
                'zlen_change', 'zwei_change', 'zwfl_change', 'zbmi_change']
for col in derived_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

print(f"✓ Filled NaN in derived features with 0")

remaining_nan = df.isna().sum().sum()
print(f"✓ Remaining NaN values: {remaining_nan}")

if remaining_nan > 0:
    print("Still have NaN values, filling all with 0...")
    df = df.fillna(0)
    print(f"✓ Final NaN count: {df.isna().sum().sum()}")

print(f"✓ Clean dataset: {len(df):,} records")

NaN counts before cleaning:
zlen                        4531
zwei                        4531
zwfl                        9883
zbmi                        4531
height_velocity_monthly    57684
weight_velocity_monthly    57684
zlen_change                61760
zwei_change                61760
zwfl_change                68669
zbmi_change                61760
dtype: int64

✓ Dropped 0 rows with NaN in critical columns
✓ Filled NaN in derived features with 0
✓ Remaining NaN values: 0
✓ Clean dataset: 310,850 records


In [4]:
# 2. CREATE SEQUENCE DATA (AUTOREGRESSIVE - FAST NUMPY VECTORIZED)

print("\n2. CREATING TIME-SERIES SEQUENCES (FAST NUMPY VECTORIZED)")
print("-" * 80)

MIN_HISTORY = 1  
MAX_SEQUENCE_LENGTH = 10  

input_features = [
    'age_months', 'height', 'weight', 'cbmi',
    'zlen', 'zwei', 'zwfl', 'zbmi',
    'gender_numeric', 'height_velocity_monthly',
    'weight_velocity_monthly', 'measurement_number'
]

target_features = ['height', 'weight']

print("Forcing all input features to be numeric...")
for col in input_features:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df = df.sort_values(['child_id', 'date']).reset_index(drop=True)

print("Converting to NumPy arrays for rapid slicing...")
input_mat = df[input_features].values.astype(np.float32)
target_mat = df[target_features].values.astype(np.float32)
child_ids = df['child_id'].values
ages = df['age_months'].values
genders = df['gender_numeric'].values

_, start_indices = np.unique(child_ids, return_index=True)
start_indices = np.sort(start_indices)
start_indices = np.append(start_indices, len(df))

sequences = []

print("Generating sequences using pure NumPy matrix slicing...")
for i in tqdm(range(len(start_indices) - 1), desc="Processing children"):
    start_row = start_indices[i]
    end_row = start_indices[i+1]
    
    n_measurements = end_row - start_row
    
    if n_measurements < 2:
        continue
        
    c_id = child_ids[start_row]
    
    c_inputs = input_mat[start_row:end_row]
    c_targets = target_mat[start_row:end_row]
    c_ages = ages[start_row:end_row]
    c_genders = genders[start_row:end_row]
    
    for h in range(MIN_HISTORY, n_measurements):
        seq_start = max(0, h - MAX_SEQUENCE_LENGTH)
        
        input_seq = c_inputs[seq_start:h]
        target_seq = c_targets[h]
        
        if np.isnan(input_seq).any() or np.isnan(target_seq).any():
            continue 
            
        sequences.append({
            'child_id': c_id,
            'input_sequence': input_seq,
            'target': target_seq,
            'target_age': c_ages[h],
            'sequence_length': len(input_seq),
            'gender': c_genders[h]
        })

print(f"\n✓ Generated {len(sequences):,} pure autoregressive training sequences in seconds.")


2. CREATING TIME-SERIES SEQUENCES (FAST NUMPY VECTORIZED)
--------------------------------------------------------------------------------
Forcing all input features to be numeric...
Converting to NumPy arrays for rapid slicing...
Generating sequences using pure NumPy matrix slicing...


Processing children: 100%|██████████| 57684/57684 [00:00<00:00, 66862.72it/s]


✓ Generated 253,166 pure autoregressive training sequences in seconds.


In [5]:
# 3. PREPARE ARRAYS FOR MODELING

print("\n3. PREPARING ARRAYS FOR MODELING")
print("-" * 80)

def pad_sequence(seq, max_len, n_features):
    if len(seq) < max_len:
        padding = np.zeros((max_len - len(seq), n_features))
        return np.vstack([padding, seq])
    else:
        return seq[-max_len:]

n_features = len(input_features)

X_padded = np.array([
    pad_sequence(s['input_sequence'], MAX_SEQUENCE_LENGTH, n_features)
    for s in sequences
]).astype(np.float32)

y = np.array([s['target'] for s in sequences]).astype(np.float32)
sequence_lengths = np.array([s['sequence_length'] for s in sequences])
target_ages = np.array([s['target_age'] for s in sequences])
child_ids = np.array([s['child_id'] for s in sequences])
genders = np.array([s['gender'] for s in sequences])

print(f"✓ Prepared arrays:")
print(f"  X shape: {X_padded.shape} (samples, timesteps, features)")
print(f"  y shape: {y.shape} (samples, targets)")
print(f"  Sequence lengths: {sequence_lengths.shape}")

# CRITICAL: FINAL INVALID VALUE CHECK AND FIX

print(f"X_padded - NaN count: {np.isnan(X_padded).sum()}")
print(f"X_padded - Inf count: {np.isinf(X_padded).sum()}")
print(f"y - NaN count: {np.isnan(y).sum()}")
print(f"y - Inf count: {np.isinf(y).sum()}")

X_padded = np.nan_to_num(X_padded, nan=0.0, posinf=1e6, neginf=-1e6)
y = np.nan_to_num(y, nan=0.0, posinf=1e6, neginf=-1e6)

X_padded = np.clip(X_padded, -1e6, 1e6)
y = np.clip(y, 1.0, 1e6)  

print(f"\n✓ After cleaning:")
print(f"  X_padded - NaN: {np.isnan(X_padded).any()}, Inf: {np.isinf(X_padded).any()}")
print(f"  y - NaN: {np.isnan(y).any()}, Inf: {np.isinf(y).any()}")
print(f"  X range: [{X_padded.min():.2f}, {X_padded.max():.2f}]")
print(f"  y range: [{y.min():.2f}, {y.max():.2f}]")


3. PREPARING ARRAYS FOR MODELING
--------------------------------------------------------------------------------
✓ Prepared arrays:
  X shape: (253166, 10, 12) (samples, timesteps, features)
  y shape: (253166, 2) (samples, targets)
  Sequence lengths: (253166,)
X_padded - NaN count: 0
X_padded - Inf count: 0
y - NaN count: 0
y - Inf count: 0

✓ After cleaning:
  X_padded - NaN: False, Inf: False
  y - NaN: False, Inf: False
  X range: [-30.44, 549.80]
  y range: [1.00, 195.00]


In [6]:
# 4. TRAIN/VALIDATION/TEST SPLIT

print("\n4. SPLITTING DATA")
print("-" * 80)

unique_children = np.unique(child_ids)
n_children = len(unique_children)

train_val_children, test_children = train_test_split(
    unique_children,
    test_size=0.15,
    random_state=42
)

train_children, val_children = train_test_split(
    train_val_children,
    test_size=0.15/0.85,  
    random_state=42
)

train_mask = np.isin(child_ids, train_children)
val_mask = np.isin(child_ids, val_children)
test_mask = np.isin(child_ids, test_children)

X_train = X_padded[train_mask]
y_train = y[train_mask]
lengths_train = sequence_lengths[train_mask]

X_val = X_padded[val_mask]
y_val = y[val_mask]
lengths_val = sequence_lengths[val_mask]

X_test = X_padded[test_mask]
y_test = y[test_mask]
lengths_test = sequence_lengths[test_mask]

print(f"✓ Data split completed:")
print(f"\nChildren split:")
print(f"  Train: {len(train_children):,} children")
print(f"  Validation: {len(val_children):,} children")
print(f"  Test: {len(test_children):,} children")

print(f"\nSequences split:")
print(f"  Train: {len(X_train):,} sequences")
print(f"  Validation: {len(X_val):,} sequences")
print(f"  Test: {len(X_test):,} sequences")


4. SPLITTING DATA
--------------------------------------------------------------------------------
✓ Data split completed:

Children split:
  Train: 40,378 children
  Validation: 8,653 children
  Test: 8,653 children

Sequences split:
  Train: 177,322 sequences
  Validation: 38,016 sequences
  Test: 37,828 sequences


In [7]:
# 5. FEATURE SCALING

print("\n5. SCALING FEATURES")
print("-" * 80)

X_train_reshaped = X_train.reshape(-1, n_features)

scaler = StandardScaler()
scaler.fit(X_train_reshaped)

def scale_sequences(X, scaler):
    original_shape = X.shape
    X_reshaped = X.reshape(-1, n_features)
    X_scaled = scaler.transform(X_reshaped)
    return X_scaled.reshape(original_shape)

X_train_scaled = scale_sequences(X_train, scaler)
X_val_scaled = scale_sequences(X_val, scaler)
X_test_scaled = scale_sequences(X_test, scaler)

print("✓ Features scaled using StandardScaler")

with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✓ Saved scaler to: ../data/processed/scaler.pkl")



5. SCALING FEATURES
--------------------------------------------------------------------------------
✓ Features scaled using StandardScaler
✓ Saved scaler to: ../data/processed/scaler.pkl


In [8]:
# 6. SAVE PREPARED DATA

print("\n6. SAVING PREPARED DATA")
print("-" * 80)

np.savez_compressed('../data/processed/train_data.npz',
                    X=X_train_scaled, y=y_train, lengths=lengths_train)

np.savez_compressed('../data/processed/val_data.npz',
                    X=X_val_scaled, y=y_val, lengths=lengths_val)

np.savez_compressed('../data/processed/test_data.npz',
                    X=X_test_scaled, y=y_test, lengths=lengths_test)

print("✓ Saved training data to: ../data/processed/train_data.npz")
print("✓ Saved validation data to: ../data/processed/val_data.npz")
print("✓ Saved test data to: ../data/processed/test_data.npz")

metadata = {
    'input_features': input_features,
    'target_features': target_features,
    'n_features': n_features,
    'n_targets': len(target_features),
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'min_history': MIN_HISTORY,
    'train_children': len(train_children),
    'val_children': len(val_children),
    'test_children': len(test_children),
    'train_sequences': len(X_train),
    'val_sequences': len(X_val),
    'test_sequences': len(X_test)
}

with open('../data/processed/metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("✓ Saved metadata to: ../data/processed/metadata.pkl")


6. SAVING PREPARED DATA
--------------------------------------------------------------------------------
✓ Saved training data to: ../data/processed/train_data.npz
✓ Saved validation data to: ../data/processed/val_data.npz
✓ Saved test data to: ../data/processed/test_data.npz
✓ Saved metadata to: ../data/processed/metadata.pkl


In [9]:
# 7. DATA STATISTICS

print("\n7. DATA STATISTICS")
print("-" * 80)

print("\nTarget Variable Statistics (Training Set):")
print(f"Height (cm):")
print(f"  Mean: {y_train[:, 0].mean():.2f} ± {y_train[:, 0].std():.2f}")
print(f"  Range: [{y_train[:, 0].min():.1f}, {y_train[:, 0].max():.1f}]")

print(f"\nWeight (kg):")
print(f"  Mean: {y_train[:, 1].mean():.2f} ± {y_train[:, 1].std():.2f}")
print(f"  Range: [{y_train[:, 1].min():.1f}, {y_train[:, 1].max():.1f}]")


7. DATA STATISTICS
--------------------------------------------------------------------------------

Target Variable Statistics (Training Set):
Height (cm):
  Mean: 87.16 ± 12.76
  Range: [40.0, 195.0]

Weight (kg):
  Mean: 12.18 ± 3.57
  Range: [1.0, 100.0]
